in this notebook we consider the preference of our llm judge (`qwen3.5:4b`) comparing our original transformer model and our DPO optimised model. the DPO optimisation process converged fairly well with good accuracy and a drop in loss; but whether this generalises to another sample of prompts and completions is yet to be seen.

In [1]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import os
import json
import copy
import random

from ollama import chat
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer
from pathlib import Path
from typing import List


# import local libs
cwd = Path.cwd()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "src").is_dir())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.generate import generate, clean, pretty_print
from src.models.transformer_lm import GPTNeo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

carry forward llm-as-a-judge utils and functions from previous notebook

In [4]:
class Preference(BaseModel):
    completion_id: int

def extract_prompt(text: str, delimiters: List[str] = ['.', '?', '!']) -> str:
    indices = [text.find(c) for c in delimiters if text.find(c) != -1]

    assert len(indices) != 0

    first_idx = min(indices)
    return text[:first_idx+1]

def build_prompt(object: dict) -> str:
    # copy then shuffle so position doesn't line up with completion_id
    completions = object['completions'][:]
    random.shuffle(completions)
    completions = " ".join(f"[Completion {c['completion_id']}] {c['completion']}" for c in completions)

    return (
        "You are judging completions of children's stories for ages 3-4, where simple words and plots are expected and are not a weakness. "
        "Choose the best of the four completions that continue the story beginning, judging in this order of importance: "
        "1) coherence: keeps the same characters, has no contradictions or repetition, and stays on the story's topic; "
        "2) grammar: correct sentences and a proper ending; "
        "3) creativity: something interesting or warm happens that a young child would enjoy. "
        "A plain coherent story beats a creative one that falls apart, and length should not affect your choice. "
        "Respond with the completion_id of the best completion. "
        f"[Story beginning] {object['prompt']} {completions}"
    )

def preference(object: dict, model: str = 'qwen3.5:4b') -> dict:
    prompt = build_prompt(object)
    response = chat(model, messages=[{'role': 'user', 'content': prompt}], think=False, format=Preference.model_json_schema(), options={ 'temperature': 0 })

    choice = Preference.model_validate_json(response.message.content)
    return {'prompt_id': object['prompt_id'], 'completion_id': choice.completion_id}

we take another sample of 500 prompts, create completions from the original transformer and post-trained transformer and keep a running tally of which completion the llm preferred. we expect our DPO trained model to be preferred over the long run. *if* we are able to validate this, this does not suggest that the post-trained model's stories are more creative, gramatically correct, etc; we could instead be influencing the model weights to incorporate biases from Qwen 3.5 4b.

either way, the Qwen model's depth and data, dominates our transformer model, and thus distilling some information and some bias from it sounds like a good thing to me?

In [5]:
# lost test data
data_path = 'C:/data/tinystories/data'
data_files = os.listdir(data_path)
valid_files = [f for f in data_files if 'validation' in f]
tst_data = pd.concat([pd.read_parquet(f'{data_path}/{v}') for v in valid_files])

# pull in model, weights and tokeniser
original_model = GPTNeo(4, 10_000, 768, 512, [3024] * 4, [8] * 4, dropout=[0] * 4).to(device)
dpo_model = GPTNeo(4, 10_000, 768, 512, [3024] * 4, [8] * 4, dropout=[0] * 4).to(device)

original_state = torch.load('C:/dev/storyteller/weights/20260926_3024_4x_8_4x_0_4x_transformer.pt')
dpo_state = torch.load('C:/dev/storyteller/weights/20260927_3024_4x_8_4x_0_4x_transformer_dpo.pt')

original_model.load_state_dict(original_state)
dpo_model.load_state_dict(dpo_state)
tokeniser = AutoTokenizer.from_pretrained('C:/dev/storyteller/weights/gpt-neo-10k-tokeniser')

tst_data['prompt'] = tst_data['text'].apply(lambda x: extract_prompt(x))
# first story is a bit weird
tst_data = tst_data[1:]

# sample 500 prompts; these could be overlaps with the training but the completions should be random
dpo_tst = tst_data.sample(500).reset_index(drop=True)

we use the same prompt to completions architecture but limit the model to two completions for each prompt - one from the tuned and untuned models. the original model will be completion id 0, and the tuned model will be completion id 1.

In [10]:
tst_completions = []
for idx, row in dpo_tst.iterrows():
    print(f'prompt :: {row.prompt} :: ({round((idx / len(dpo_tst)* 100), 3)}%)')
    tst_completions.append({
        'prompt_id': len(tst_completions),
        'prompt': row.prompt,
        'completions': [
            {'completion_id': 0, 'completion': generate(row.prompt, original_model, tokeniser, device)},
            {'completion_id': 1, 'completion': generate(row.prompt, dpo_model, tokeniser, device)}
        ]
    })

prompt :: Lily and Ben are friends. :: (0.0%)
prompt :: Once upon a time, there was a little dog named Max. :: (0.2%)
prompt :: Once upon a time, there was a little girl named Lily. :: (0.4%)
prompt :: Once upon a time, there was a girl called Julia. :: (0.6%)
prompt :: Once upon a time, there was a computer. :: (0.8%)
prompt :: Lila and Ben are twins. :: (1.0%)
prompt :: Once upon a time, there was an old oak tree. :: (1.2%)
prompt :: Once upon a time, there was a little girl named Lily. :: (1.4%)
prompt :: Once upon a time there was a mom and a dad that loved each other very much. :: (1.6%)
prompt :: Once upon a time, there was a big and strong lion. :: (1.8%)
prompt :: Tommy was a silly little boy. :: (2.0%)
prompt :: Once upon a time, there was a boy named Timmy. :: (2.2%)
prompt :: Once upon a time, there was a little girl named Lily. :: (2.4%)
prompt :: Once upon a time, there was a little girl named Lily. :: (2.6%)
prompt :: Once upon a time, there was a mule named Max. :: (2.8%

In [14]:
tst_preferences = []

with ThreadPoolExecutor(max_workers=12) as pool:
    for idx, result in enumerate(pool.map(preference, tst_completions)):
        print(f'judged :: {result} :: ({round((idx + 1) / len(tst_completions) * 100, 3)}%)')
        tst_preferences.append(result)

judged :: {'prompt_id': 0, 'completion_id': 1} :: (0.2%)
judged :: {'prompt_id': 1, 'completion_id': 0} :: (0.4%)
judged :: {'prompt_id': 2, 'completion_id': 1} :: (0.6%)
judged :: {'prompt_id': 3, 'completion_id': 1} :: (0.8%)
judged :: {'prompt_id': 4, 'completion_id': 1} :: (1.0%)
judged :: {'prompt_id': 5, 'completion_id': 0} :: (1.2%)
judged :: {'prompt_id': 6, 'completion_id': 1} :: (1.4%)
judged :: {'prompt_id': 7, 'completion_id': 0} :: (1.6%)
judged :: {'prompt_id': 8, 'completion_id': 0} :: (1.8%)
judged :: {'prompt_id': 9, 'completion_id': 0} :: (2.0%)
judged :: {'prompt_id': 10, 'completion_id': 0} :: (2.2%)
judged :: {'prompt_id': 11, 'completion_id': 0} :: (2.4%)
judged :: {'prompt_id': 12, 'completion_id': 1} :: (2.6%)
judged :: {'prompt_id': 13, 'completion_id': 0} :: (2.8%)
judged :: {'prompt_id': 14, 'completion_id': 1} :: (3.0%)
judged :: {'prompt_id': 15, 'completion_id': 0} :: (3.2%)
judged :: {'prompt_id': 16, 'completion_id': 0} :: (3.4%)
judged :: {'prompt_id': 

In [27]:
from scipy.stats import binomtest

tst_preferences_dt = pd.DataFrame(tst_preferences)
tst_preferences_dt['trained_better'] = tst_preferences_dt['completion_id'] == 1

valid = tst_preferences_dt[tst_preferences_dt['completion_id'].isin([0, 1])]
wins = int(valid['trained_better'].sum())
n = len(valid)

result = binomtest(wins, n, p=0.5, alternative='two-sided')
ci = result.proportion_ci(confidence_level=0.95)

print(f'win rate :: {round(wins / n * 100, 2)}% ({wins}/{n})')
print(f'p-value :: {round(result.pvalue, 4)}')
print(f'95% ci :: {round(ci.low * 100, 2)}% - {round(ci.high * 100, 2)}%')

win rate :: 56.2% (281/500)
p-value :: 0.0063
95% ci :: 51.73% - 60.6%


our trained model is preferred by the Qwen model 56.2% of the time, which is significant at the $\alpha = 5\%$. this suggests that the model learnt in DPO optimisation to incorporate the model's views on story creativity, accuracy and coherence - but is it possible to see a *qualitative* difference?

In [17]:
print('Original ::')
pretty_print(generate('Once upon a time, there was a boy named Lucas', original_model, tokeniser, device))
print('\nDPO post-trained ::')
pretty_print(generate('Once upon a time, there was a boy named Lucas', dpo_model, tokeniser, device))

Original ::
Once upon a time, there was a boy named Lucas. He was very angry and frustrated
and wanted to strike something cheap. He spotted a pencil on the floor and
picked it up. He took it and walked over to his mom. "Mom," he yelled, "I want
to strike this paper!" His mom smiled. She gave Lucas back the pencil and said,
"But you have to be careful. Ice can be sharp and can hurt you." Lucas nodded
and thanked his mom. He put the pencil in his pocket and kept exploring the Tom
game. He knew there was no more scary but fun game to play.

DPO post-trained ::
Once upon a time, there was a boy named Lucas. His favorite thing to do was to
play baseball. Lucas was very good at baseball and he always wanted to practice
when he got home. One day, Lucas went outside to play baseball. He was so
excited because he had never driven a ball the day before! He started to run and
kick the ball, and he was having a lot of fun! Soon, Lucas stopped running and
sat in the sun. He was so tired from all h

In [20]:
print('Original ::')
pretty_print(generate('Nick was a silly boy, who loved to play outside.', original_model, tokeniser, device))
print('\nDPO post-trained ::')
pretty_print(generate('Nick was a silly boy, who loved to play outside.', dpo_model, tokeniser, device))

Original ::
Nick was a silly boy, who loved to play outside. One day, as Nick and his mom
were walking to the park, Nick saw something unusual. It was an ashtray on the
ground. Nick knew that it was made from mud, from then on, with a stinky spot.
His mom was surprised too â€“ why not everyone should use a nasty old ashtray in
their garden. Nick decided to take the ashtray home with him. When he got home,
the next day, his mom went back to the ashtray to clean it up. She asked Nick to
help her in pumpp yuckying at the smelly ashtray. Nick was a bit scared, as he
did not know how to save the ashtray. His mom kindly said â€œWe should do this,
it is important to stay in the garden and work together with the ashtray.
Letâ€™s put it in a clean jar.â€ Nick was very glad and said â€œOkay, Iâ€™m
going to keep my ashtray.â€ He carefully placed it in a jar and went back to the
garden. He knew it was important not to use drains near its dirty house.

DPO post-trained ::
Nick was a silly boy, who 

In [23]:
print('Original ::')
pretty_print(generate('Connor was an angry boy.', original_model, tokeniser, device))
print('\nDPO post-trained ::')
pretty_print(generate('Connor was an angry boy.', dpo_model, tokeniser, device))

Original ::
Connor was an angry boy. He had seen it and he was embarrassed. "You should,' he
said. you should always be a good young boy, no matter what," stayed his father
from then on. From then on, Frian was a very thoughtful mother!

DPO post-trained ::
Connor was an angry boy. "Roar!" he exclaimed. "What are you honking for? Juice
Connor!" Claire was surprised. She had never met a car before. "Oice Connor!"
she quacked. And the boy smiled. "Can I honk it too?" Claire nodded and the boy
began to Moo. She and the boy laughed as they said hello. Soon they were all
laughing and playing together. It was a great day because the boy was always
clapping and Ellie was glowing with joy.


qualitatively, the post-trained model seems to stay on track of the story more, as well as create more coherent stories. for example, when prompted with an angry boy, the story continues with a story line about his anger - whereas the original model sort of hallucinates a story out of thin air. both models clearly still struggle when there is something more novel, like a reference to a sport - it appears to amalgamate different knowledges of different sports; 'he had never driven the ball', 'he started to run and kick the ball' which is clearly not well attended to, considering he sport is `baseball`.

conclusively however, i think that the DPO optimisation does aid in creating more coherent stories. done at a larger scale, post-training likely helps align the model to create more interesting and vibrant stories (text).